# LADA on Google Colab — Notebook 1 / 2: TRAIN

Training run for **LADA: Scalable Label-Specific CLIP Adapter for Continual Learning** (ICML 2025)
on the **X-TAIL** benchmark, in the **16-shot, Order-I** setting (the canonical X-TAIL setting — paper Tables 1 & 7).
Repo: https://github.com/maolinluo/lada

**This notebook trains the 10-task continual-learning chain and saves a checkpoint snapshot after _each_ task to Drive.**
The companion **eval notebook** (`lada_colab_eval.ipynb`) then loads those snapshots to reproduce the per-step
accuracy matrix (Table 7) and dump per-sample predictions + task routing (for Figure 3) — without retraining.

**Before you start:** `Runtime > Change runtime type > Hardware accelerator = GPU`.

Run modes (set in the **Config** cell):
- **`quick`** — 3 small datasets (Caltech101 + EuroSAT + MNIST). Fits on **free Colab**. Pipeline smoke test, ~15-25 min.
- **`full`** — the full 10-dataset benchmark (the paper's run). Needs **~80 GB disk** → use **Drive** (`USE_DRIVE=True`).

Why save per-step checkpoints: each task overwrites a single `checkpoint.pth.tar` (it's cumulative — holds the
LADA features + classifier + prototypes for all tasks so far). We copy each step's file to Drive so a Colab
disconnect doesn't cost the per-step state, and so the eval notebook can reconstruct the full matrix.
The run is also **resumable** — re-running skips steps whose snapshot already exists.


## 1. Check the GPU


In [1]:
!nvidia-smi
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0),
          '| total mem GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1))
else:
    raise SystemExit('No GPU! Set Runtime > Change runtime type > GPU, then re-run.')


Sat May 23 09:47:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config
Pick the run mode and where to store the dataset.


In [ ]:
# === LADA replication run: 16-shot, Order-I (X-TAIL) — paper Tables 1 & 7 ===
MODE = 'full'        # 'full' = all 10 datasets (the paper's benchmark). 'quick' = 3-dataset smoke test.
USE_DRIVE = True     # Recommended True: datasets + per-step checkpoints persist across Colab disconnects.
SHOTS = 16           # 16-shot setting.
ORDER = 'I'          # task order. Order-I (alphabetical) is the must-have run.

RUN_NAME = f'TAIL_{SHOTS}shot_order{ORDER}'   # artifact folder name on Drive
OUTPUT_DIR = 'TAIL_colab'                     # local ./output dir (single cumulative checkpoint = the chain state)
print('MODE =', MODE, '| USE_DRIVE =', USE_DRIVE, '| SHOTS =', SHOTS, '| ORDER =', ORDER, '| RUN_NAME =', RUN_NAME)


## 3. Clone the repo and install dependencies
Colab already ships a CUDA-enabled PyTorch, so we only install the remaining requirements.


In [ ]:
%cd /content
![ -d lada ] || git clone https://github.com/maolinluo/lada.git
%cd /content/lada
!pip -q install -r requirements.txt modelscope
print('done')


## 4. Patch the code (tqdm progress bar)
Adds a tqdm bar to the training loop (upstream only prints a line every 10 batches).
**No GPU cap** — on Colab the GPU is dedicated, so LADA uses the whole card. Idempotent — safe to re-run.


In [ ]:
from pathlib import Path

# --- trainer.py: add a tqdm progress bar to the training loop ---
t = Path('trainer.py').read_text()

old_loop = ('            num_batches = len(self.train_loader)\n'
            '            for batch_idx, batch in enumerate(self.train_loader):\n'
            '                data_time.update(time.time() - end)')
new_loop = ('            num_batches = len(self.train_loader)\n'
            '            pbar = tqdm(self.train_loader, total=num_batches, ascii=True, leave=False,\n'
            '                        desc=f"Train {cfg.dataset} epoch [{epoch_idx + 1}/{num_epochs}]")\n'
            '            for batch_idx, batch in enumerate(pbar):\n'
            '                data_time.update(time.time() - end)')
if new_loop not in t:
    assert old_loop in t, 'training loop not found (upstream changed?)'
    t = t.replace(old_loop, new_loop)

old_pf = ('                batch_time.update(time.time() - end)\n\n'
          '                meet_freq = (batch_idx + 1) % cfg.print_freq == 0')
new_pf = ('                batch_time.update(time.time() - end)\n\n'
          '                pbar.set_postfix(loss=f"{loss_meter.avg:.4f}",\n'
          '                                 acc=f"{acc_meter.avg:.2f}",\n'
          '                                 lr=f"{current_lr:.2e}")\n\n'
          '                meet_freq = (batch_idx + 1) % cfg.print_freq == 0')
if 'set_postfix' not in t:
    assert old_pf in t
    t = t.replace(old_pf, new_pf)

t = t.replace('                    print(" ".join(info))',
              '                    tqdm.write(" ".join(info))')
Path('trainer.py').write_text(t)
print('patched trainer.py (tqdm only). No GPU cap -> LADA uses the full Colab GPU.')


## 5. Download the X-TAIL dataset
Datasets come from ModelScope. Each zip is downloaded, extracted, then **deleted to save disk**.
`quick` mode pulls only 3 small datasets; `full` pulls all 10 (Sun397 alone is ~36 GB).


In [ ]:
import os, subprocess

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_ROOT = '/content/drive/MyDrive/Project/LADA/X-TAIL'   # <- your pre-uploaded datasets
else:
    DATASET_ROOT = '/content/datasets/X-TAIL'
os.makedirs(DATASET_ROOT, exist_ok=True)
print('DATASET_ROOT =', DATASET_ROOT)

QUICK = ['Caltech101.zip', 'EuroSAT.zip', 'MNIST.zip']
FULL  = ['Aircraft.zip', 'Caltech101.zip', 'DTD.zip', 'EuroSAT.zip', 'Flowers.zip',
         'Food.zip', 'MNIST.zip', 'Pets.zip', 'StanfordCars.zip', 'Sun397.zip']
zips = QUICK if MODE == 'quick' else FULL

# All 10 dataset folders already exist on Drive -> this loop just skips (no download).
for z in zips:
    out_dir = os.path.join(DATASET_ROOT, z[:-4])  # zip extracts to a dir of the same name
    if os.path.isdir(out_dir):
        print(f'[skip] {z} (already extracted)')
        continue
    print(f'[download] {z} ...')
    subprocess.run(['modelscope', 'download', '--dataset', 'ForestLuo/X-TAIL',
                    '--local_dir', DATASET_ROOT, z], check=True)
    print(f'[unzip] {z} ...')
    subprocess.run(['unzip', '-q', '-o', os.path.join(DATASET_ROOT, z), '-d', DATASET_ROOT], check=True)
    os.remove(os.path.join(DATASET_ROOT, z))  # free disk
    print(f'[done] {z}')

print('\nExtracted datasets:', sorted(d for d in os.listdir(DATASET_ROOT) if os.path.isdir(os.path.join(DATASET_ROOT, d))))
!df -h /content | tail -1


## 6. Point the config at the dataset
Writes `dataset_sequence` (3 datasets for quick, all 10 for full) and the dataset `root` into the config.


In [ ]:
seq_quick = ['caltech101', 'eurosat', 'mnist']
seq_full  = ['aircraft', 'caltech101', 'dtd', 'eurosat', 'flowers',
             'food101', 'mnist', 'oxford_pets', 'stanford_cars', 'sun397']
seq = seq_quick if MODE == 'quick' else seq_full

with open('configs/data/TAIL.yaml', 'w') as f:
    f.write('dataset_sequence: %s\n' % seq)
    f.write('root: "%s"\n' % DATASET_ROOT)

print(open('configs/data/TAIL.yaml').read())


## 7. Train + snapshot each task, then aggregate
Runs each task sequentially (continual learning). **After every task** the cumulative `checkpoint.pth.tar`
is copied to Drive as `step{NN}_{dataset}.pth.tar` — that's what the eval notebook consumes.
You'll see a **tqdm bar per epoch** with live loss/acc. `quick` is fast; `full` takes a few hours.

**Resumable:** if the session drops, just re-run this cell — steps whose snapshot already exists on Drive are
skipped, and the last snapshot is restored as the chain checkpoint so the next task continues correctly.
At the end it prints the **per-step accuracy matrix + Transfer/Average/Last means** (Table 7) and syncs logs to Drive.


In [ ]:
import os, shutil, subprocess

NUM_WORKERS = os.cpu_count() or 8   # use all available CPU cores for data loading
print('CPU cores / num_workers =', NUM_WORKERS)

# Where to persist artifacts. With USE_DRIVE=True these live on Drive and survive disconnects.
RUN_BASE = (f'/content/drive/MyDrive/Project/LADA/runs/{RUN_NAME}' if USE_DRIVE
            else f'/content/lada/runs/{RUN_NAME}')
CKPT_DIR = os.path.join(RUN_BASE, 'checkpoints')
LOG_DIR  = os.path.join(RUN_BASE, 'logs')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
print('Run artifacts ->', RUN_BASE)

LOCAL_OUT = os.path.join('output', OUTPUT_DIR)              # main.py writes here
CKPT_SRC  = os.path.join(LOCAL_OUT, 'checkpoint.pth.tar')   # cumulative chain checkpoint

# (dataset, num_epochs, continue_train_first) -- epochs match the authors' run_TAIL_16shot.sh (Order-I)
runs_quick = [('caltech101', 10, True), ('eurosat', 100, False), ('mnist', 200, False)]
runs_full  = [('aircraft', 40, True), ('caltech101', 10, False), ('dtd', 30, False),
              ('eurosat', 100, False), ('flowers', 30, False), ('food101', 5, False),
              ('mnist', 200, False), ('oxford_pets', 10, False), ('stanford_cars', 30, False),
              ('sun397', 10, False)]
runs = runs_quick if MODE == 'quick' else runs_full

for k, (ds, ep, first) in enumerate(runs, start=1):
    snap = os.path.join(CKPT_DIR, f'step{k:02d}_{ds}.pth.tar')
    if os.path.exists(snap):
        # Already done in a previous (possibly disconnected) session -> restore as the chain
        # checkpoint so the NEXT task's continue_train load picks up the right state.
        os.makedirs(LOCAL_OUT, exist_ok=True)
        shutil.copy2(snap, CKPT_SRC)
        print(f'[skip step {k:02d}] {ds}: snapshot exists -> restored as chain checkpoint')
        continue

    cmd = ['python', 'main.py', '-d', 'TAIL', '-m', 'clip_vit_b16',
           'num_shots', str(SHOTS), 'dataset', ds, 'num_epochs', str(ep),
           'num_workers', str(NUM_WORKERS), 'output_dir', OUTPUT_DIR]
    if first:
        cmd += ['continue_train_first', 'True']
    print('\n>>>', ' '.join(cmd))
    subprocess.run(cmd, check=True)

    # Snapshot this step's cumulative checkpoint (the canonical file stays in place for the chain).
    shutil.copy2(CKPT_SRC, snap)
    print(f'[saved step {k:02d}] {ds} -> {snap}')

print('\n==== FINAL RESULTS (Table 7 / Transfer-Average-Last means) ====')
subprocess.run(['python', 'result_process.py', '-d', 'TAIL', '--output_dir', OUTPUT_DIR], check=True)

# Sync per-task logs + result.txt to Drive (logs are the matrix source; survive disconnects).
for fn in os.listdir(LOCAL_OUT):
    if fn.startswith('log_') or fn == 'result.txt':
        shutil.copy2(os.path.join(LOCAL_OUT, fn), os.path.join(LOG_DIR, fn))
print('\nSynced logs + result.txt to', LOG_DIR)
print('Checkpoints on Drive:', sorted(os.listdir(CKPT_DIR)))
print('\n*** RUN COMPLETE when you see all 10 step checkpoints listed above. ***')


## Notes & troubleshooting
- **Disk full in `full` mode on free Colab** — expected; Sun397 pushes peak usage past ~80 GB. Use Colab **Pro**, or set `USE_DRIVE = True` (needs a Google One plan with enough space).
- **Session disconnects** — free Colab caps sessions (~12 h) and idle time. `full` is long; keep the tab active or use Pro. With `USE_DRIVE=True` the dataset survives a reconnect, so you only re-run from step 4.
- **`quick` results are not comparable to the paper** — it's a 3-dataset smoke test to confirm the pipeline works. Use `full` for the real benchmark numbers.
- **Order-II** — swap `seq_full` for the order-II sequence and reorder `runs_full` accordingly (see `scripts/run_TAIL_16shot_order2.sh`).
